In [1]:
## Import modules
import os, sys
import numpy as np
import geopandas as gpd
import cftime
import gc
import shapely
import json
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets
import matplotlib.pyplot as plt
import glob
# Import Plotly for interactive plotting
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as pc
from ipywidgets import interact, IntSlider, Dropdown, VBox, HBox
import ipywidgets as widgets
import psutil
import time 
import numba
numba.set_num_threads(2)

# Add the directory containing 'cmct' to the Python path
# Navigate two levels up to reach main CmCt dir
cmct_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))

# Initialising Logger
import logging

logging.basicConfig(
    level=logging.ERROR, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Import utilities for this comparison
sys.path.insert(0, cmct_dir)
from cmct.time_utils import *
from cmct.calving import *
from cmct.calving_modules.interpolation import *
from cmct.calving_modules.residual_calculation import *
# from cmct.calving_modules.json_to_netcdf import *
# from cmct.shapefile_utils import *

# Force initial garbage collection
gc.collect()


40

In [2]:
# Reload modules to pick up any changes to imports
import importlib
import cmct.calving
import cmct.calving_modules.residual_calculation
from cmct.calving_modules.plotting_utils import *
from cmct.calving import calculate_basin_statistics, format_basin_stats
from cmct.calving import calculate_basin_statistics

importlib.reload(cmct.calving)
importlib.reload(cmct.calving_modules.residual_calculation)

# Re-import to ensure functions are available
from cmct.calving import *

# Memory monitoring utility
def get_memory_usage():
    """Get current memory usage in MB"""
    import psutil
    process = psutil.Process()
    return process.memory_info().rss / 1024 / 1024

def log_memory_usage(stage=""):
    """Log current memory usage"""
    try:
        mem_usage = get_memory_usage()
        print(f"Memory usage {stage}: {mem_usage:.1f} MB")
    except ImportError:
        print("psutil not available for memory monitoring")
    except Exception as e:
        print(f"Error checking memory: {e}")

# Check initial memory usage
log_memory_usage("at start")


Memory usage at start: 268.3 MB


In [3]:
# Observation Dataset
# Ice sheet
loc = "GIS"  # 'GIS' or 'AIS'

# Set the observation data dir path
obs_filename = cmct_dir + "/data/calving/observed_icemask_ismip_annual.nc"

# To use aggregation functions for basin
basin_aggregation = True  # IMPORTANT

basin_filename = (
    cmct_dir + "/bin/Calving/GRE_Basins_IMBIE2_v1.3/GRE_Basins_IMBIE2_v1.3.shp"
)

# Set the Model Data dir path
# model_filename = cmct_dir + "/test/calving/ensemble/sftgif_B001_hist.nc"
model_files = cmct_dir + "/test/calving/ensemble/*.nc"

# Set time range for comparison
start_year = 2007
end_year = 2009

# List of basins (ex ["NW", "NE"]) to compare if all -> "all", if none -> False
# If you do not know which basins are in the model, you can put "auto"
# NOTE: Align this list with the basins in the model.
basin_list = ["NW"]

# Output filetype and filename
filetype = "netcdf"  # netcdf or json or None
filename = "calving_comparison"

# Optional Configurations
interpolation_method = "slinear"  # 'nearest', 'linear', 'cubic'
accuracy_calculation_method = "mean"  # 'mean', 'RMS',

colors = {
    "CW": "blue",
    "NE": "red",
    "SE": "green",
    "SW": "orange",
    "NO": "purple",
    "NW": "brown",
}

MAX_CPU = 80
MAX_MEM = 80


In [4]:
# Check if observation file exist
if not os.path.exists(obs_filename):
    raise FileNotFoundError(f"Observation file not found: {obs_filename}")


if basin_aggregation and not os.path.exists(basin_filename):
    raise FileNotFoundError(f"Basin shapefile not found: {basin_filename}")
    # Load basin shapes

print(basin_filename)
basins, basin_list = load_basins(basin_filename, basin_list)

print(obs_filename)
gsfc = load_gsfc_calving(obs_filename, basins)
gsfc.ds["time"] = standardising_time_var(gsfc.time)

gsfc_stats = calculate_gsfc_statistics(gsfc, basins)



/Users/aditya_pachpande/Documents/GitHub/CmCt/bin/Calving/GRE_Basins_IMBIE2_v1.3/GRE_Basins_IMBIE2_v1.3.shp
/Users/aditya_pachpande/Documents/GitHub/CmCt/data/calving/observed_icemask_ismip_annual.nc


In [5]:
print(gsfc_stats)

{'NW': {np.float64(1973.0): {'count': 270870, 'mean': np.float64(0.9933462177428287), 'std': np.float64(0.06185225411429242), 'min': np.float64(0.0), 'max': np.float64(1.0), 'rms': np.float64(0.9952700184588111), 'rss': np.float64(268313.6399), 'sum': np.float64(269067.69), 'winsorized_mean': 1.0, 'outlier_weighted_mean': np.float64(0.9999792034698074)}, np.float64(1974.0): {'count': 270870, 'mean': np.float64(0.993339350980175), 'std': np.float64(0.061893163400958004), 'min': np.float64(0.0), 'max': np.float64(1.0), 'rms': np.float64(0.9952657081812338), 'rss': np.float64(268311.3159), 'sum': np.float64(269065.83), 'winsorized_mean': 1.0, 'outlier_weighted_mean': np.float64(0.9999791842488035)}, np.float64(1975.0): {'count': 270870, 'mean': np.float64(0.9933344777937758), 'std': np.float64(0.06189719576123881), 'min': np.float64(0.0), 'max': np.float64(1.0), 'rms': np.float64(0.995261095199113), 'rss': np.float64(268308.82869999995), 'sum': np.float64(269064.51000000007), 'winsorized_

In [6]:
model_files = glob.glob(model_files)
# Sort the files for consistent ordering
model_files.sort()
# Generate model names from filenames (extract basename without extension)
model_names = [os.path.splitext(os.path.basename(f))[0] for f in model_files]

# Validate that we found model files
if not model_files:
    raise FileNotFoundError(
        f"No model files found matching pattern: {model_files}"
    )

print(f"Found {len(model_files)} model files:")
for file in model_files:
    print(os.path.basename(file))

Found 5 model files:
sftgif_B001_hist.nc
sftgif_B002_hist.nc
sftgif_B003_hist.nc
sftgif_B004_hist.nc
sftgif_B005_hist.nc


In [ ]:
def process_single_model_with_mask(
    file, gsfc, basin_mask, basin_names, x_coords, y_coords, start_year, end_year
):
    """
    Process a single model file using pre-computed basin mask.

    Parameters
    ----------
    file : str
        Path to the model file
    gsfc : GSFCcalving
        GSFC calving data object
    basin_mask : np.ndarray
        Pre-computed basin mask
    basin_names : list
        List of basin names
    x_coords : np.ndarray
        X coordinates
    y_coords : np.ndarray
        Y coordinates
    start_year : int
        Start year for analysis
    end_year : int
        End year for analysis

    Returns
    -------
    dict
        Basin statistics for the model
    """
    print(f"Processing: {file}")

    try:
        model_res = load_model_calving(file)

        # Validate model data
        if model_res is None or model_res.ds is None:
            raise ValueError(f"Failed to load model data from {file}")

        model_res.ds["time"] = standardising_time_var(model_res.time)

        # Handle Time Range with validation
        try:
            checking_calving_daterange(
                gsfc.time.values, model_res.time.values, start_year, end_year
            )
        except Exception as e:
            print(f"Warning: Time range validation failed: {e}")

        # Interpolation with error handling
        try:
            interpolater = Interpolater(model_res, gsfc)
            model_res.ds = interpolater.interpolate()
            print(f"Resampled data shape: {model_res.ds.dims}")
        except Exception as e:
            print(f"Warning: Interpolation failed: {e}")
            # Continue with original resolution if interpolation fails

        years = np.arange(start_year, end_year + 1)

        # Create residuals dataset using pre-computed basin mask
        try:
            residuals_dataset = create_calving_dataset_with_mask(
                gsfc, model_res, years, basin_mask, basin_names, x_coords, y_coords
            )

            # Validate dataset
            if residuals_dataset is None:
                raise ValueError("Failed to create residuals dataset")

        except Exception as e:
            print(f"Error creating residuals dataset: {e}")
            # Return empty stats if dataset creation fails
            return {
                year: {"NW": {"count": 0, "mean": np.nan, "std": np.nan}}
                for year in years
            }

        # Load residuals with error handling
        try:
            residuals = load_residuals(residuals_dataset)
            if residuals is None:
                raise ValueError("Failed to load residuals")
        except Exception as e:
            print(f"Error loading residuals: {e}")
            return {
                year: {"NW": {"count": 0, "mean": np.nan, "std": np.nan}}
                for year in years
            }

        # Calculate basin statistics with error handling
        try:
            basin_stats = calculate_basin_statistics(residuals)
            print(format_basin_stats(basin_stats))
        except Exception as e:
            print(f"Error calculating basin statistics: {e}")
            basin_stats = {
                year: {"NW": {"count": 0, "mean": np.nan, "std": np.nan}}
                for year in years
            }

        # Memory cleanup
        del model_res, interpolater, residuals_dataset, residuals
        gc.collect()

        return basin_stats

    except Exception as e:
        print(f"Error processing model file {file}: {e}")
        # Return fallback statistics
        years = np.arange(start_year, end_year + 1)
        return {
            year: {"NW": {"count": 0, "mean": np.nan, "std": np.nan}} for year in years
        }


In [ ]:
basin_stats_array = []
failed_files = []

print(f"Processing {len(model_files)} model files...")

for i, file in enumerate(model_files):
    try:
        if not os.path.exists(file):
            raise FileNotFoundError(f"Model file not found: {file}")
            
        print(f"\n[{i+1}/{len(model_files)}] Processing model file: {os.path.basename(file)}")
        
        # Process with timeout and memory monitoring
        basin_stats = process_single_model_with_mask(file, gsfc, basin_mask, basin_names, x_coords, y_coords, start_year, end_year)

        # Validate the returned statistics
        if basin_stats and any(year in basin_stats for year in range(start_year, end_year + 1)):
            basin_stats_array.append(basin_stats)
            print(f"Successfully processed {os.path.basename(file)}")
        else:
            print(f"Invalid statistics returned for {os.path.basename(file)}")
            failed_files.append(file)
        
        del basin_stats
        gc.collect()
        
        cpu = psutil.cpu_percent()
        mem = psutil.virtual_memory().percent
        
        if cpu > MAX_CPU or mem > MAX_MEM:
            print("High resource usage, waiting...")
            time.sleep(5)
        
    except Exception as e:
        print(f"Error processing model file {os.path.basename(file)}: {e}")
        failed_files.append(file)
        
        # Force cleanup on error
        gc.collect()
        continue

Processing 5 model files...

[1/5] Processing model file: sftgif_B001_hist.nc
Processing: /Users/aditya_pachpande/Documents/GitHub/CmCt/test/calving/ensemble/sftgif_B001_hist.nc
The selected dates 2007 to 2009 are within the overlapping data range.
Resampled data shape: FrozenMappingWarningOnValuesAccess({'time': 14, 'x': 1680, 'y': 2880})


2025-07-24 13:07:52,214 - ERROR - Basin mask shape (1680, 2880) doesn't match expected (2880, 1680)
2025-07-24 13:07:52,214 - ERROR - This indicates the coordinate transformation didn't work as expected


=== Statistics for Year 2007 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 415.95709228515625 |   270870 |  0.00153563 |  0.00158687 |  0.00001312 |   0.056317 |   0.056338
-------------------------------------------------------------------------------------


=== Statistics for Year 2008 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 448.8782653808594 |   270870 |  0.00165717 |  0.00171128 |  0.00001562 |   0.056234 |   0.056258
-------------------------------------------------------------------------------------


=== Statistics for Year 2009 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
------------------------------------------------------------------------

2025-07-24 13:08:17,208 - ERROR - Basin mask shape (1680, 2880) doesn't match expected (2880, 1680)
2025-07-24 13:08:17,208 - ERROR - This indicates the coordinate transformation didn't work as expected


=== Statistics for Year 2007 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 415.95709228515625 |   270870 |  0.00153563 |  0.00158687 |  0.00001312 |   0.056317 |   0.056338
-------------------------------------------------------------------------------------


=== Statistics for Year 2008 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 448.8782653808594 |   270870 |  0.00165717 |  0.00171128 |  0.00001562 |   0.056234 |   0.056258
-------------------------------------------------------------------------------------


=== Statistics for Year 2009 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
------------------------------------------------------------------------

...

=== Statistics for Year 2007 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 415.95709228515625 |   270870 |  0.00153563 |  0.00158687 |  0.00001312 |   0.056317 |   0.056338
-------------------------------------------------------------------------------------


=== Statistics for Year 2008 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 448.8782653808594 |   270870 |  0.00165717 |  0.00171128 |  0.00001562 |   0.056234 |   0.056258
-------------------------------------------------------------------------------------


=== Statistics for Year 2009 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
------------------------------------------------------------------------

2025-07-24 13:09:07,271 - ERROR - Basin mask shape (1680, 2880) doesn't match expected (2880, 1680)
2025-07-24 13:09:07,271 - ERROR - This indicates the coordinate transformation didn't work as expected


=== Statistics for Year 2007 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 415.95709228515625 |   270870 |  0.00153563 |  0.00158687 |  0.00001312 |   0.056317 |   0.056338
-------------------------------------------------------------------------------------


=== Statistics for Year 2008 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 448.8782653808594 |   270870 |  0.00165717 |  0.00171128 |  0.00001562 |   0.056234 |   0.056258
-------------------------------------------------------------------------------------


=== Statistics for Year 2009 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
------------------------------------------------------------------------

2025-07-24 13:09:31,938 - ERROR - Basin mask shape (1680, 2880) doesn't match expected (2880, 1680)
2025-07-24 13:09:31,939 - ERROR - This indicates the coordinate transformation didn't work as expected


=== Statistics for Year 2007 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 415.95709228515625 |   270870 |  0.00153563 |  0.00158687 |  0.00001312 |   0.056317 |   0.056338
-------------------------------------------------------------------------------------


=== Statistics for Year 2008 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
NW    | 448.8782653808594 |   270870 |  0.00165717 |  0.00171128 |  0.00001562 |   0.056234 |   0.056258
-------------------------------------------------------------------------------------


=== Statistics for Year 2009 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
------------------------------------------------------------------------

In [9]:
if basin_stats_array:
    print(f"First model stats keys: {list(basin_stats_array[0].keys())}")
    print(f"Sample basin stats for first model: {basin_stats_array[0]}")

First model stats keys: [np.int64(2007), np.int64(2008), np.int64(2009)]
Sample basin stats for first model: {np.int64(2007): {np.str_('NW'): {'count': 270870, 'mean': np.float32(0.0015356337), 'std': np.float32(0.0563172), 'min': np.float32(-1.0), 'max': np.float32(1.0), 'rms': np.float32(0.056338128), 'rss': np.float32(859.73724), 'sum': np.float32(415.9571), 'winsorized_mean': np.float64(0.0015868740301593946), 'outlier_weighted_mean': np.float32(1.3117617e-05)}}, np.int64(2008): {np.str_('NW'): {'count': 270870, 'mean': np.float32(0.0016571723), 'std': np.float32(0.056233827), 'min': np.float32(-1.0), 'max': np.float32(1.0), 'rms': np.float32(0.056258246), 'rss': np.float32(857.3009), 'sum': np.float32(448.87827), 'winsorized_mean': np.float64(0.0017112816254955842), 'outlier_weighted_mean': np.float32(1.562134e-05)}}, np.int64(2009): {np.str_('NW'): {'count': 270870, 'mean': np.float32(0.001719219), 'std': np.float32(0.056942336), 'min': np.float32(-1.0), 'max': np.float32(1.0), '

# Working with the Basin Statistics Array

In [10]:
def process_model_for_raw_data(file):
    """
    Process a single model file to extract raw model values (not residuals)
    """
    print(f"Processing model for raw data: {file}")
    model_res = load_model_calving(file)
    model_res.ds["time"] = standardising_time_var(model_res.time)

    # Handelling Time Range
    checking_calving_daterange(
        gsfc.time.values, model_res.time.values, start_year, end_year
    )
    interpolater = Interpolater(model_res, gsfc)
    model_res.ds = interpolater.interpolate()
    print(f"\nResampled data shape: {model_res.ds.dims}")
    
    years = np.arange(start_year, end_year + 1)
    
    # Instead of creating residuals, calculate statistics directly from model data
    model_stats = {}
    
    for year in years:
        model_stats[year] = {}
        
        # Get data for this year
        year_data = model_res.ds.sel(time=str(year))
        
        for basin_name in basin_list:
            # Get basin mask
            basin_geom = basins[basins['NAME'] == basin_name].geometry.iloc[0]
            
            # Create mask for this basin
            mask = create_basin_mask(year_data, basin_geom)
            
            # Extract model values within the basin
            basin_data = year_data.sftgif.values[mask]
            
            if len(basin_data) > 0:
                model_stats[year][basin_name] = {
                    'mean': float(np.nanmean(basin_data)),
                    'std': float(np.nanstd(basin_data)),
                    'count': int(np.sum(~np.isnan(basin_data))),
                    'min': float(np.nanmin(basin_data)),
                    'max': float(np.nanmax(basin_data))
                }
            else:
                model_stats[year][basin_name] = {
                    'mean': np.nan,
                    'std': np.nan,
                    'count': 0,
                    'min': np.nan,
                    'max': np.nan
                }
    
    del model_res, interpolater
    gc.collect()
    
    return model_stats

def create_basin_mask(data, basin_geom):
    """
    Create a boolean mask for a basin geometry
    """
    # Get coordinates
    lons, lats = np.meshgrid(data.x.values, data.y.values)
    
    # Create points
    from shapely.geometry import Point
    points = [Point(lon, lat) for lon, lat in zip(lons.flatten(), lats.flatten())]
    
    # Check which points are in the basin
    mask = np.array([basin_geom.contains(point) for point in points])
    
    return mask.reshape(lons.shape)

# Ensemble Plotting and Analysis

In [11]:
# Reload the plotting module to ensure latest changes are available
import importlib
import cmct.calving_modules.plotting_utils
importlib.reload(cmct.calving_modules.plotting_utils)

# Import ensemble plotting utilities
from cmct.calving_modules.plotting_utils import (
    create_ensemble_time_series_plot,
    create_interactive_ensemble_plot,
    create_ensemble_statistics_summary,
)

In [12]:
# Create interactive ensemble plot without GSFC line
print("Creating interactive ensemble time series plot (models only)...")
print(f"Models: {model_names}")
print(f"Basins: {basin_list}")

# Create the interactive plot without GSFC
ensemble_plot_widget = create_interactive_ensemble_plot(
    basin_stats_array, 
    model_names, 
    basin_list=basin_list, 
    gsfc_stats=None  # Remove GSFC line
)

# Display the widget
ensemble_plot_widget

Creating interactive ensemble time series plot (models only)...
Models: ['sftgif_B001_hist', 'sftgif_B002_hist', 'sftgif_B003_hist', 'sftgif_B004_hist', 'sftgif_B005_hist']
Basins: ['NW']


In [13]:
# Create a new plot comparing the sum of all models with GSFC
print("\nCreating model sum vs GSFC comparison plot...")

def calculate_ensemble_sum(basin_stats_array, basin_list=None):
    """
    Calculate the sum of all models for each basin and year
    
    Parameters
    ----------
    basin_stats_array : list
        List of basin statistics from all models
    basin_list : list, optional
        List of basin names to process
        
    Returns
    -------
    dict
        Dictionary with ensemble sum statistics
    """
    if not basin_stats_array:
        return {}
    
    # Get years and basins
    years = sorted(list(basin_stats_array[0].keys()))
    if basin_list is None:
        basin_list = list(basin_stats_array[0][years[0]].keys())
    
    ensemble_sum = {}
    
    # Calculate sum for each basin and year
    for basin_name in basin_list:
        ensemble_sum[basin_name] = {}
        
        for year in years:
            # Collect values from all models for this basin/year
            model_values = []
            for basin_stats in basin_stats_array:
                if (year in basin_stats and 
                    basin_name in basin_stats[year] and 
                    basin_stats[year][basin_name]["count"] > 0):
                    model_values.append(basin_stats[year][basin_name]["mean"])
            
            if model_values:
                # Store the sum and other statistics
                ensemble_sum[basin_name][year] = {
                    "mean": sum(model_values),  # Sum of all models
                    "std": np.std(model_values),
                    "count": len(model_values),
                    "model_count": len(model_values)
                }
    
    return ensemble_sum

def create_sum_vs_gsfc_plot(ensemble_sum, gsfc_stats, basin_list, statistic="mean"):
    """
    Create a plot comparing ensemble sum with GSFC observations
    """
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    
    # Create subplot with one plot per basin
    fig = make_subplots(
        rows=len(basin_list),
        cols=1,
        subplot_titles=[f"Basin {basin} - Model Sum vs GSFC" for basin in basin_list],
        shared_xaxes=True,
        vertical_spacing=0.02,
    )
    
    years = sorted(list(ensemble_sum[basin_list[0]].keys()))
    
    # Plot each basin
    for basin_idx, basin_name in enumerate(basin_list):
        row = basin_idx + 1
        
        # Get ensemble sum values
        sum_values = []
        for year in years:
            if year in ensemble_sum[basin_name]:
                sum_values.append(ensemble_sum[basin_name][year][statistic])
            else:
                sum_values.append(np.nan)
        
        # Add ensemble sum trace
        fig.add_trace(
            go.Scatter(
                x=years,
                y=sum_values,
                mode="lines+markers",
                name="Model Sum" if basin_idx == 0 else None,
                line=dict(color="blue", width=3),
                marker=dict(size=8, color="blue"),
                showlegend=(basin_idx == 0),
                hovertemplate="<b>Model Sum</b><br>"
                + f"Basin: {basin_name}<br>"
                + "Year: %{x}<br>"
                + f"Sum: %{{y:.6f}}<extra></extra>",
            ),
            row=row,
            col=1,
        )
        
        # Add GSFC trace if available
        if gsfc_stats is not None and basin_name in gsfc_stats:
            gsfc_values = []
            for year in years:
                if year in gsfc_stats[basin_name]:
                    gsfc_values.append(gsfc_stats[basin_name][year][statistic])
                else:
                    gsfc_values.append(np.nan)
            
            fig.add_trace(
                go.Scatter(
                    x=years,
                    y=gsfc_values,
                    mode="lines+markers",
                    name="GSFC" if basin_idx == 0 else None,
                    line=dict(color="red", width=3),
                    marker=dict(size=8, color="red"),
                    showlegend=(basin_idx == 0),
                    hovertemplate="<b>GSFC</b><br>"
                    + f"Basin: {basin_name}<br>"
                    + "Year: %{x}<br>"
                    + f"GSFC: %{{y:.6f}}<extra></extra>",
                ),
                row=row,
                col=1,
            )
    
    # Update layout
    fig.update_layout(
        title=f"Ensemble Model Sum vs GSFC Comparison",
        height=300 * len(basin_list),
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    
    # Update axes
    fig.update_xaxes(title_text="Year", row=len(basin_list), col=1)
    for i in range(len(basin_list)):
        fig.update_yaxes(title_text="Ice Mask Value", row=i + 1, col=1)
    
    return fig

# Calculate ensemble sum
ensemble_sum_stats = calculate_ensemble_sum(basin_stats_array, basin_list)

# Create and display the comparison plot
if ensemble_sum_stats:
    sum_comparison_fig = create_sum_vs_gsfc_plot(
        ensemble_sum_stats, 
        gsfc_stats, 
        basin_list
    )
    sum_comparison_fig.show()
    
    # Print some summary statistics
    print(f"\nEnsemble Sum Summary:")
    for basin_name in basin_list:
        if basin_name in ensemble_sum_stats:
            years = list(ensemble_sum_stats[basin_name].keys())
            if years:
                latest_year = max(years)
                latest_sum = ensemble_sum_stats[basin_name][latest_year]["mean"]
                model_count = ensemble_sum_stats[basin_name][latest_year]["model_count"]
                print(f"  {basin_name}: Latest sum ({latest_year}): {latest_sum:.6f} (from {model_count} models)")
else:
    print("No ensemble sum data available for plotting")


Creating model sum vs GSFC comparison plot...



Ensemble Sum Summary:
  NW: Latest sum (2009): 0.008568 (from 5 models)


In [ ]:
# Create interactive version of the sum comparison plot
print("\nCreating interactive model sum vs GSFC comparison...")

def create_interactive_sum_comparison(ensemble_sum_stats, gsfc_stats, basin_list):
    """
    Create an interactive comparison plot with dropdown for statistic selection
    """
    import ipywidgets as widgets
    
    # Available statistics
    stats_options = [
        ("Mean", "mean"),
        ("Standard Deviation", "std"),
    ]
    
    # Create dropdown widget
    stat_dropdown = widgets.Dropdown(
        options=stats_options,
        value="mean",
        description="Statistic:",
        style={"description_width": "initial"},
    )
    
    # Create output widget for plot
    output = widgets.Output()
    
    def update_plot(change):
        with output:
            output.clear_output(wait=True)
            fig = create_sum_vs_gsfc_plot(
                ensemble_sum_stats,
                gsfc_stats,
                basin_list,
                statistic=change["new"]
            )
            fig.show()
    
    # Initial plot
    with output:
        fig = create_sum_vs_gsfc_plot(
            ensemble_sum_stats,
            gsfc_stats,
            basin_list,
            statistic="mean"
        )
        fig.show()
    
    # Connect dropdown to update function
    stat_dropdown.observe(update_plot, names="value")
    
    return widgets.VBox([stat_dropdown, output])

# Create and display the interactive widget
interactive_sum_widget = create_interactive_sum_comparison(
    ensemble_sum_stats, 
    gsfc_stats, 
    basin_list
)

print("Interactive Model Sum vs GSFC Comparison:")
interactive_sum_widget


Creating interactive model sum vs GSFC comparison...


Interactive Model Sum vs GSFC Comparison:
